# 水果图片分类

随着计算机视觉技术的快速发展，图像分类已成为人工智能领域的重要应用之一。水果图片分类作为图像分类的一个具体应用场景，不仅具有实际的商业价值（如水果自动分拣、质量检测等），也为学习和实践深度学习技术提供了良好的实验平台。本次实训项目旨在通过 MindSpore 框架和残差网络（ResNet）结构，实现对多种水果图片的高效分类。

## 1. 项目背景介绍

图像分类是最基础的计算机视觉应用，属于有监督学习类别，如给定一张图像(猫、狗、飞机、汽车等等)，判断图像所属的类别。   
卷积神经网络（CNN）在图像分类方面处理能力很强，可自行对图像进行特征提取。  
本项目使用残差网络结构对水果数据集进行分类。

- 文件列表   
    该数据集包含 1 个文件，33 个子文件夹，16854 张图片   
    - train  
        - Apple Braeburn  
        - ...  
        - Watermelon  

- 属性描述   
    `train`文件下的 33 个子文件夹的名称就是该类型水果图片的名称

在此项目中，我们将探索如何借助 mindsporeAPI，通过残差网络结构对水果图片分类

## 2. 数据探索和可视化

- 导入所需的库

In [ ]:
import mindspore as ms
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import mindspore.nn as nn
import mindspore.ops as ops
from typing import List, Optional

from mindspore.common.initializer import Normal
from mindspore import dtype as mstype
from mindspore.dataset import ImageFolderDataset
from PIL import Image
from mindspore import load_checkpoint, load_param_into_net


In [ ]:
data_dir = "/home/jovyan/work/datasets/6892bfae9e78856c69427a90-momodel/train"
seed = 1234
random.seed(seed)
np.random.seed(seed)
ms.set_seed(seed)


- 获取`train`文件下的 33 个子文件名以及图片数量

In [ ]:
files_names=os.listdir(data_dir)
files_nums=[]
for file_name in files_names:
    path=data_dir+'/'+file_name
    file_num=len(os.listdir(path))
    files_nums.append(file_num)
files_names

- 对各水果及其图片数量可视化

In [ ]:
plt.figure(figsize=(15, 10))
x_width = range(0,len(files_names))
color=['red','black']
plt.rcParams["font.sans-serif"]=['SimHei']
plt.rcParams["axes.unicode_minus"]=False
plt.barh(x_width,files_nums,lw=0.3,color=color,height=0.5)
plt.yticks(range(0,len(files_names)),files_names)
for i,j in zip(files_nums,x_width):
   plt.text(i+10,j,"%d"%i,ha="center",va="bottom")
plt.title("水果数量")
plt.ylabel("水果名称")
plt.xlabel("数量")
plt.show()


可以发现，各类别图片的数量主要集中在 450~500 之间，除极个别类型图片的数量较多

- 显示水果样例图片及其 shape

In [ ]:
test_image = data_dir+'/'+files_names[1]+'/'+f'{files_names[1]}_0.jpg'
img = np.array(Image.open(test_image))
print(f"Data Shape = {img.shape}")
plt.imshow(img)
plt.title("水果图片")
plt.ylabel("Y")
plt.xlabel("X")
plt.show()


- 随机显示 64 张样例水果图片

In [ ]:
def display_random_images(image_data_array, N):
    grid_size = int(np.ceil(np.sqrt(N)))
    selected_images = np.random.choice(image_data_array.shape[0], N, replace=False)
    fig, axs = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    for i, ax in enumerate(axs.flatten()):
        if i < N:
            ax.imshow(image_data_array[selected_images[i]])
            ax.axis('off')
        else:
            fig.delaxes(ax)
    plt.tight_layout()
    plt.show()


In [ ]:
# 每组水果种类中选择4个样本图片
max_img_num=4
img_data_array = np.empty((len(files_names)*max_img_num, 100, 100, 3), dtype=int)
index=0
for foldername in files_names:
    path=data_dir+'/'+foldername
    for filename in os.listdir(path):
        img_path=path+'/'+filename
        img = Image.open(img_path)
        img_array = np.asarray(img)
        img_data_array[index] = img_array
        index+=1
        if index%max_img_num==0:
            break


In [ ]:
display_random_images(img_data_array, 64)

## 3. 模型训练

- 属于残差网络结构进行特征提取加图片分类
    - 其特点是可以实现搭建较深的网络结构，网络层数越深，其训练误差和测试误差越小

- 数据集加载
    - 使用 `mindspore.dataset.ImageFolderDataset` 函数进行读取数据集，并将其打乱
    - 对图像进行图像处理

In [ ]:
def create_dataset(dataset_dir, usage, resize, batch_size, workers,num_samples,shuffle=True,decode=True,class_indexing=None,crop_size=32):

    data_set = ImageFolderDataset(
        dataset_dir=dataset_dir, shuffle=shuffle, decode=decode,num_samples=num_samples,num_parallel_workers=workers, class_indexing=class_indexing)
    trans = []
    if usage == "train":
        trans += [
            vision.RandomCrop((crop_size,crop_size), (4, 4, 4, 4)),
            vision.RandomHorizontalFlip(prob=0.5)
        ]
    trans += [
        vision.Resize(resize),
        vision.Rescale(1.0 / 255.0, 0.0),
        vision.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010]),
        vision.HWC2CHW()
    ]

    target_trans = transforms.TypeCast(mstype.int32)

    data_set = data_set.map(operations=trans,
                            input_columns='image',
                            num_parallel_workers=workers)

    data_set = data_set.map(operations=target_trans,
                            input_columns='label',
                            num_parallel_workers=workers)

    data_set = data_set.batch(batch_size)
    return data_set


- 获取训练数据集

In [ ]:
classes=files_names #分类大小
batch_size = 20  # 批量大小
re_image_size =32  # 训练图像空间大小
crop_size=100  # 随机切片大小
workers = 4  # 并行线程个数
num_classes =len(classes)  # 分类数量
num_samples=8000 #训练图片大小
dataset_train =create_dataset(dataset_dir=data_dir,
                                       usage="train",
                                       resize=re_image_size,
                                       batch_size=batch_size,
                                        crop_size=crop_size,
                                       workers=workers,
                                       num_samples=num_samples)
step_size_train =dataset_train.get_dataset_size()


In [ ]:
step_size_train


- 获取测试数据集

In [ ]:
num_samples=1000
dataset_val =create_dataset(dataset_dir=data_dir,
                                     usage="test",
                                     resize=re_image_size,
                                     batch_size=batch_size,
                                     workers=workers,
                                     num_samples=num_samples)
step_size_val = dataset_val.get_dataset_size()


In [ ]:
step_size_val


- 构建网络

残差网络由两个分支构成：主分支和 shortcuts，主分支是经过堆叠一系列卷积操作得出，shortcuts 从输入直接输出    
在最后两个特征矩阵相加，在通过`Relu`激活函数后，即为最后的输出结果

- 此项目使用的结构图如下，主分支有四层卷积网络结构:   
    - 第一层 以输入 channel 为 128 为例，通过数量为 64，大小为 1 \* 1 的卷积核进行降维，然后是 Normalization 层，最后通过`Relu`激活函数层,输出 channel 为 64
    - 第二层 通过数量为 64，大小为 3 \* 3，步长为 1 的卷积核提取特征，然后是 Normalization 层，最后通过`Relu`激活函数层,输出 channel 为 64
    - 第三层 通过数量为 64，大小为 3 \* 3，步长为 stride 的卷积核提取特征，然后是 Normalization 层，最后通过`Relu`激活函数层,输出 channel 为 64
    - 第四层 通过数量为 128，大小为 1 \* 1 的卷积核进行升维，然后是 Normalization 层,输出 channel 为 128
    - 主分支的特征矩阵与 shortcuts 输出的特征矩阵相加，通过`Relu`激活函数后输出

In [ ]:
weight_init = Normal(mean=0, sigma=0.02)
gamma_init = Normal(mean=1, sigma=0.02)
class ResidualNetwork(nn.Cell):
    expansion = 4  # 最后一个卷积核的数量是第一个卷积核数量的4倍
    def __init__(self, in_channel: int, out_channel: int,
                 stride: int = 1, down_sample: Optional[nn.Cell] = None) -> None:
        super(ResidualNetwork, self).__init__()

        self.conv1 = nn.Conv2d(in_channel, out_channel,
                               kernel_size=1, weight_init=weight_init)
        self.norm1 = nn.BatchNorm2d(out_channel)
        self.conv2 = nn.Conv2d(out_channel, out_channel,
                               kernel_size=3, stride=1,
                               weight_init=weight_init)
        self.norm2 = nn.BatchNorm2d(out_channel)
        self.conv3 = nn.Conv2d(out_channel, out_channel,
                               kernel_size=3, stride=stride,
                               weight_init=weight_init)
        self.norm3 = nn.BatchNorm2d(out_channel)
        self.conv4 = nn.Conv2d(out_channel, out_channel * self.expansion,
                               kernel_size=1, weight_init=weight_init)
        self.norm4 = nn.BatchNorm2d(out_channel * self.expansion)
        self.relu = nn.ReLU()
        self.down_sample = down_sample
    def construct(self, x):

        identity = x  # shortscuts分支

        out = self.conv1(x)  # 主分支第一层：1*1卷积层
        out = self.norm1(out)
        out = self.relu(out)
        out = self.conv2(out)  # 主分支第二层：3*3卷积层
        out = self.norm2(out)
        out = self.relu(out)
        out = self.conv3(out)  # 主分支第三层：3*3卷积层
        out = self.norm3(out)
        out = self.relu(out)
        out = self.conv4(out)  # 主分支第四层：1*1卷积层
        out = self.norm4(out)
        if self.down_sample is not None:
            identity = self.down_sample(x)

        out += identity  # 输出为主分支与shortcuts之和
        out = self.relu(out)

        return out


- 构建残差块

In [ ]:
def make_layer(last_out_channel, block,channel: int, block_nums: int, stride: int = 1):
        down_sample = None  # shortcuts分支
        if stride != 1 or last_out_channel != channel * block.expansion:
            down_sample = nn.SequentialCell([
                nn.Conv2d(last_out_channel, channel * block.expansion,
                          kernel_size=1, stride=stride, weight_init=weight_init),
                nn.BatchNorm2d(channel * block.expansion, gamma_init=gamma_init)
            ])
        layers = []
        layers.append(block(last_out_channel, channel, stride=stride, down_sample=down_sample))
        in_channel = channel * block.expansion
        # 堆叠残差网络
        for _ in range(1, block_nums):
            layers.append(block(in_channel, channel))
        return nn.SequentialCell(layers)


- 此 ResNet 网络有 6 个卷积结构，一个平均池化层，一个全连接层
    - 第一层： 输入图片大小为 32×32，输入 channel为 3 最后输出 feature map 大小为 16×16，channel 为 64
    - 第二层： 输入 feature map 大小为 16×16，输入 channel 为 64，该层堆叠 3 个 [1×1,64;3×3,64;3×3,64;1×1,256] 最后输出 feature map 大小为 8x8，channel 为 256
    - 第三层： 输入 feature map 大小为 8×8，输入 channel 为 256，经过一个卷积核为 3×3，步长为 1 的操作, 该层堆叠 3 个 [1×1,128;3×3,128;3×3,128;1×1,512] 最后输出 feature map 大小为 8×8，channel 为 512
    - 第四层： 输入 feature map 大小为 8×8，输入 channel 为 512，经过一个卷积核为 3×3，步长为 2 的操作, 该层堆叠 4 个 [1×1,256;3×3,256;3×3,256;1×1,1024] 最后输出 feature map 小为 4×4，channel 为 1024
    - 第五层： 输入 feature map 大小为 4×4，输入 channel 为 1024，经过一个卷积核为 3×3，步长为 2 的操作, 该层堆叠 3 个 [1×1,512;3×3,512;3×3,512;1×1,2048] 最后输出 feature map 大小为 2×2，channel 为 2048
    - 第六层： 输入 feature map 大小为 2×2，输入 channel 为 2048，经过一个卷积核为 3×3，步长为 2 的操作,该层堆叠 2 个 [1×1,1024;3×3,1024;3×3,1024;1×1,4096] 最后输出 feature map 大小为 1×1，channel 为 4096

In [ ]:
class ResNet(nn.Cell):
    def __init__(self, block,
                 layer_nums: List[int], num_classes: int, input_channel: int) -> None:
        super(ResNet, self).__init__()
        self.relu = nn.ReLU()
        # 第一个卷积层，输入channel为3，输出channel为64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, weight_init=weight_init)
        self.norm = nn.BatchNorm2d(64)
        # 最大池化层，缩小图片的尺寸
        self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, pad_mode='same')
        # 各个残差网络结构块定义
        # 第二个卷积层，输入channel为64，输出channel为256
        self.layer1 = make_layer(64, block, 64, layer_nums[0])
        # 第三个卷积层，输入channel为256，输出channel为512
        self.layer2 = make_layer(64 * block.expansion, block, 128, layer_nums[1], stride=1)
        # 第三个卷积层，输入channel为512，输出channel为1024
        self.layer3 = make_layer(128 * block.expansion, block, 256, layer_nums[2], stride=2)
        # 第四个卷积层，输入channel为1024，输出channel为2048
        self.layer4 = make_layer(256 * block.expansion, block, 512, layer_nums[3], stride=2)
        # 第四个卷积层，输入channel为2048，输出channel为4096
        self.layer5 = make_layer(512 * block.expansion, block, 1024, layer_nums[4], stride=2)
        # 平均池化层
        self.avg_pool = nn.AvgPool2d()
        # flattern层
        self.flatten = nn.Flatten()
        # 全连接层
        self.fc = nn.Dense(in_channels=input_channel, out_channels=num_classes)
    def construct(self, x):
        x = self.conv1(x)
        x = self.norm(x)
        x = self.relu(x)
        x = self.max_pool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        x = self.avg_pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x


- 模型训练
    - 使用的是 mindspore 中常用的训练方式

In [ ]:
input_channel=4096
#设置各层迭代的次数
layers=[3, 3, 4, 3,2]
# 定义ResNet50网络
network = ResNet(ResidualNetwork,layers,num_classes,input_channel)
# 全连接层输入层的大小
in_channel = network.fc.in_channels

path_names=os.listdir(data_dir)
num_classes =len(path_names)

fc = nn.Dense(in_channels=in_channel, out_channels=num_classes)
# 重置全连接层
network.fc = fc
# 设置学习率
num_epochs = 8
lr = nn.cosine_decay_lr(min_lr=0.00001, max_lr=0.001, total_step=step_size_train * num_epochs,
                        step_per_epoch=step_size_train, decay_epoch=num_epochs)
# 定义优化器和损失函数
opt = nn.Momentum(params=network.trainable_params(), learning_rate=lr, momentum=0.9)
loss_fn = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')


def forward_fn(inputs, targets):
    logits = network(inputs)
    loss = loss_fn(logits, targets)
    return loss


grad_fn = ms.value_and_grad(forward_fn, None, opt.parameters)


def train_step(inputs, targets):
    loss, grads = grad_fn(inputs, targets)
    opt(grads)
    return loss
# 创建迭代器
data_loader_train = dataset_train.create_tuple_iterator(num_epochs=num_epochs)
data_loader_val = dataset_val.create_tuple_iterator(num_epochs=num_epochs)
#模型训练
def train(data_loader, epoch):
    losses = []
    network.set_train(True)
    for i, (images, labels) in enumerate(data_loader):
        loss = train_step(images, labels)
        losses.append(loss)
    return sum(losses) / len(losses)


In [ ]:
#模型验证
def evaluate(data_loader):
    network.set_train(False)
    correct_num = 0.0  # 预测正确个数
    total_num = 0.0  # 预测总数
    for images, labels in data_loader:
        logits = network(images)
        pred = logits.argmax(axis=1)  # 预测结果
        correct = ops.equal(pred, labels).reshape((-1, ))
        correct_num += correct.sum().asnumpy()
        total_num += correct.shape[0]
    acc = correct_num / total_num  # 准确率
    return acc


- 开始循环训练  
训练代码见`main.py`， 使用 gpu 进行训练, 以下仅展示代码运行结果

In [ ]:
# 可以直接读取训练好的模型进行预测，见第4节
# 开始循环训练
best_ckpt_path = "/home/jovyan/work/results/resnet-bestnew_1.ckpt"
for epoch in range(num_epochs):
    curr_loss = train(data_loader_train, epoch)
    curr_acc = evaluate(data_loader_val)
ms.save_checkpoint(network, best_ckpt_path)


- 训练的结果

Epoch: 1 / 8, Average Train Loss: 3.845, Accuracy: 0.215  
Epoch: 2 / 8, Average Train Loss: 2.751, Accuracy: 0.293  
Epoch: 3 / 8, Average Train Loss: 1.855, Accuracy: 0.694  
Epoch: 4 / 8, Average Train Loss: 1.290, Accuracy: 0.791  
Epoch: 5 / 8, Average Train Loss: 0.924, Accuracy: 0.890  
Epoch: 6 / 8, Average Train Loss: 0.637, Accuracy: 0.918  
Epoch: 7 / 8, Average Train Loss: 0.438, Accuracy: 0.933  
Epoch: 8 / 8, Average Train Loss: 0.360, Accuracy: 0.937

## 4. 预测水果的种类

训练好的模型保存在`results`文件夹下，名称 resnet-bestnew.ckpt，模型通过`main.py`文件进行训练的

In [ ]:
def visualize_model(net, dataset_val, classes):
    data = next(dataset_val.create_dict_iterator())
    images = data["image"]
    labels = data["label"]
    # 预测图像类别
    output = net(data['image'])
    pred = np.argmax(output.asnumpy(), axis=1)
    # 显示图像及图像的预测值
    plt.figure(figsize=(8, 8))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        color = 'blue' if pred[i] == labels.asnumpy()[i] else 'red'
        plt.title('predict:{}'.format(classes[pred[i]]), color=color)
        picture_show = np.transpose(images.asnumpy()[i], (1, 2, 0))
        mean = np.array([0.4914, 0.4822, 0.4465])
        std = np.array([0.2023, 0.1994, 0.2010])
        picture_show = std * picture_show + mean
        picture_show = np.clip(picture_show, 0, 1)
        plt.imshow(picture_show)
        plt.axis('off')
    plt.show()


In [ ]:
input_channel=4096
layers=[3, 3, 4, 3,2]
num_epochs=1
#加载训练好的模型
resnet_best_ckpt='/home/jovyan/work/results/resnet-bestnew.ckpt'
model = ResNet(ResidualNetwork, layers, num_classes, input_channel)
param_dict = load_checkpoint(resnet_best_ckpt)
load_param_into_net(model, param_dict)
network = model


- 加载测试数据集

In [ ]:
num_samples=500
dataset_test =create_dataset(dataset_dir=data_dir,
                                     usage="test",
                                     resize=re_image_size,
                                     batch_size=batch_size,
                                     workers=workers,
                                     num_samples=num_samples)


In [ ]:
data_loader_test = dataset_test.create_tuple_iterator(num_epochs=1)


In [ ]:
network.set_train(False)
correct_num = 0.0  # 预测正确个数
total_num = 0.0  # 预测总数
for images, labels in data_loader_test:
    logits = network(images)
    pred = logits.argmax(axis=1)  # 预测结果
    correct = ops.equal(pred, labels).reshape((-1, ))
    correct_num += correct.sum().asnumpy()
    total_num += correct.shape[0]
acc = correct_num / total_num  # 准确率


- 测试数据集的正确率

In [ ]:
acc


- 预测前9张图片的结果展示

In [ ]:
visualize_model(network, dataset_test,classes)


## 5. 总结

可以直观的感受到，卷积神经网络可以不用人为的进行图片的特征提取，同时使用残差网络结构可以有效的抑制模型过拟合风险

另外，还有许多其他的图像分类算法如 KNN，SVM，BP 等，可以尝试使用它们，来比较其中的优缺点